In [2]:
import torch
import torch.nn as nn
import pytest
import math

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"

In [ ]:
# BPE Tokenizer

def get_stats(ids):
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = 1 + counts.get(pair, 0)
    return counts


def merge(ids, pair, idx):
    i = 0
    new_ids = []
    while i < len(ids):
        if i < len(ids) - 1 and (ids[i], ids[i+1]) == pair:
            new_ids.append(idx)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1
    return new_ids


class BPE:
    def __init__(self, vocab_size, text):
        self.vocab_size = vocab_size
        self.text = text
        self.vocab = {i : bytes([i]) for i in range(256)}
        self.ids = list(text.encode("utf-8"))
        self.merges = {}

    def train(self):
        num_merges = self.vocab_size - 256
        for i in range(num_merges):
            stats = get_stats(self.ids)
            idx = 256 + i
            pair = max(stats, key=stats.get)
            self.vocab[idx] = self.vocab[pair[0]] + self.vocab[pair[1]]
            self.merges[pair] = idx
            self.ids = merge(self.ids, pair, idx)
        return self.ids

    def decode(self, ids):
        tokens = b"".join(self.vocab[idx] for idx in ids)
        text = tokens.decode("utf-8", errors="replace")
        return text

    def encode(self, text):
        ids = list(text.encode("utf-8"))
        for pair, idx in self.merges.items():
            ids = merge(ids, pair, idx)
        return ids


bpe = BPE(vocab_size=265, text="aaabcaabcasdasdsadasdafsdgsd")
bpe.train()
encoded = bpe.encode("aaabcaabcasdasdsadasdafsdgsd")
assert bpe.decode(encoded) == "aaabcaabcasdasdsadasdafsdgsd"
assert bpe.decode(bpe.ids) == "aaabcaabcasdasdsadasdafsdgsd"  # training result too

    

In [28]:
# Basic Attention Layer


class AttentionLayer(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, d_model)
        self.Wv = nn.Linear(d_model, d_model)
        self.Wo = nn.Linear(d_model, d_model)

    def forward(self, x):
        S = x.shape[1]
        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)
        attention_scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_model)
        causal_mask = torch.triu(
            torch.ones(S, S, device=x.device, dtype=torch.bool),
            diagonal=1,
        )
        attention_scores = attention_scores.masked_fill(causal_mask, float("-inf"))
        attention_weights = nn.functional.softmax(attention_scores, dim=-1)
        A = attention_weights @ V
        return self.Wo(A)

In [29]:
# Multi-Head Attention Layer


class MultiHeadAttentionLayer(nn.Module):

    def __init__(self, d_model, num_heads):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, d_model)
        self.Wv = nn.Linear(d_model, d_model)
        self.Wo = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, S, _ = x.shape

        Q = self.Wq(x)
        Q = Q.reshape(B, S, self.num_heads, self.d_head)
        Q = Q.transpose(1, 2)
        K = self.Wk(x)
        K = K.reshape(B, S, self.num_heads, self.d_head)
        K = K.transpose(1, 2)
        V = self.Wv(x)
        V = V.reshape(B, S, self.num_heads, self.d_head)
        V = V.transpose(1, 2)
        attention_scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_head)
        causal_mask = torch.triu(
            torch.ones(S, S, device=x.device, dtype=torch.bool), diagonal=1
        )
        attention_scores = attention_scores.masked_fill(causal_mask, float("-inf"))
        attention_weights = nn.functional.softmax(attention_scores, dim=-1)
        A = (attention_weights @ V)
        A = A.transpose(1, 2)
        A = A.reshape(B, S, self.d_model)
        return self.Wo(A)

In [30]:
# Multi-Query Attention Layer


class MultiQueryAttentionLayer(nn.Module):

    def __init__(self, d_model, num_heads):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, self.d_head)
        self.Wv = nn.Linear(d_model, self.d_head)
        self.Wo = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, S, _ = x.shape

        Q = self.Wq(x)
        Q = Q.reshape(B, S, self.num_heads, self.d_head)
        Q = Q.transpose(1, 2)

        K = self.Wk(x)
        K = K.reshape(B, S, 1, self.d_head)
        K = K.transpose(1, 2)

        V = self.Wv(x)
        V = V.reshape(B, S, 1, self.d_head)
        V = V.transpose(1, 2)

        attention_scores = Q @ K.transpose(2, 3) / math.sqrt(self.d_head)

        causal_mask = torch.triu(
            torch.ones(S, S, device=x.device, dtype=torch.bool), diagonal=1
        )

        attention_scores = attention_scores.masked_fill(causal_mask, float("-inf"))

        attention_weights = nn.functional.softmax(attention_scores, dim=-1)

        A = attention_weights @ V
        A = A.transpose(1, 2)
        A = A.reshape(B, S, self.d_model)

        return self.Wo(A)

In [31]:
# Grouped Query Attention Layer


class GroupedQueryAttentionLayer(nn.Module):
    def __init__(self, d_model, num_heads, num_kv_heads):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.num_kv_heads = num_kv_heads
        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, num_kv_heads * self.d_head)
        self.Wv = nn.Linear(d_model, num_kv_heads * self.d_head)
        self.Wo = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, S, _ = x.shape

        Q = self.Wq(x)
        Q = Q.reshape(B, S, self.num_heads, self.d_head)
        Q = Q.reshape(
            B, S, self.num_kv_heads, self.num_heads // self.num_kv_heads, self.d_head
        )
        Q = Q.permute(0, 2, 3, 1, 4)

        K = self.Wk(x)
        K = K.reshape(B, S, self.num_kv_heads, self.d_head)
        K = K.reshape(B, S, 1, self.num_kv_heads, self.d_head)
        K = K.permute(0, 3, 2, 1, 4)

        V = self.Wv(x)
        V = V.reshape(B, S, self.num_kv_heads, self.d_head)
        V = V.reshape(B, S, 1, self.num_kv_heads, self.d_head)
        V = V.permute(0, 3, 2, 1, 4)

        attention_scores = Q @ K.transpose(3, 4) / math.sqrt(self.d_head)

        causal_mask = torch.triu(
            torch.ones(S, S, device=x.device, dtype=torch.bool), diagonal=1
        )

        attention_scores = attention_scores.masked_fill(causal_mask, float("-inf"))

        attention_weights = nn.functional.softmax(attention_scores, dim=-1)

        A = attention_weights @ V
        A = A.permute(0, 3, 1, 2, 4)
        A = A.reshape(B, S, self.num_heads, self.d_head)
        A = A.reshape(B, S, self.d_model)

        return self.Wo(A)

In [3]:
# Multi Latent Attention Layer


class MultiLatentAttentionLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_compress_q, d_compress_kv):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.d_compress_q = d_compress_q
        self.d_compress_kv = d_compress_kv
        self.Wdq = nn.Linear(d_model, d_compress_q)
        self.Wuq = nn.Linear(d_compress_q, d_model)
        self.Wdkv = nn.Linear(d_model, d_compress_kv)
        self.Wuk = nn.Linear(d_compress_kv, d_model)
        self.Wuv = nn.Linear(d_compress_kv, d_model)
        self.Wo = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, S, _ = x.shape

        Q = self.Wuq(self.Wdq(x))
        Q = Q.reshape(B, S, self.num_heads, self.d_head)
        Q = Q.transpose(1, 2)

        K = self.Wuk(self.Wdkv(x))
        K = K.reshape(B, S, self.num_heads, self.d_head)
        K = K.transpose(1, 2)

        V = self.Wuv(self.Wdkv(x))
        V = V.reshape(B, S, self.num_heads, self.d_head)
        V = V.transpose(1, 2)

        attention_scores = Q @ K.transpose(2, 3) / math.sqrt(self.d_head)

        causal_mask = torch.triu(
            torch.ones(S, S, device=x.device, dtype=torch.bool), diagonal=1
        )

        attention_scores = attention_scores.masked_fill(causal_mask, float("-inf"))

        attention_weights = nn.functional.softmax(attention_scores, dim=-1)

        A = attention_weights @ V
        A = A.transpose(1, 2)
        A = A.reshape(B, S, self.d_model)

        return self.Wo(A)

In [33]:
# Sinusoidal Positional Encodings


class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, seq_len, d_model):
        super().__init__()
        pe = torch.zeros(seq_len, d_model)
        pos = torch.arange(seq_len).float().unsqueeze(1)
        inv_freq = 1.0 / (10000 ** (torch.arange(0, d_model, 2).float() / d_model))
        pe[:, 0::2] = torch.sin(pos * inv_freq)
        pe[:, 1::2] = torch.cos(pos * inv_freq)
        self.register_buffer("PE", pe)

    def forward(self, x):
        return x + self.PE[:x.size(1)]

In [ ]:
# Rotary Positional Encoding 


class RotaryPositionalEncoding(nn.Module):
    def __init__(self, seq_len, d_head):
        super().__init__()
        self.d_head = d_head
        pe = torch.zeros(seq_len, d_head // 2, 2, 2)
        pos = torch.arange(seq_len).float().unsqueeze(1)
        inv_freq = 1.0 / (10000 ** (torch.arange(0, d_head, 2).float() / d_head))
        pe[:, :, 0, 0] = torch.cos(pos * inv_freq)
        pe[:, :, 0, 1] = -torch.sin(pos * inv_freq)
        pe[:, :, 1, 0] = torch.sin(pos * inv_freq)
        pe[:, :, 1, 1] = torch.cos(pos * inv_freq)
        self.register_buffer("PE", pe)

    def applyRoPE(self, Q, K):
        B, h, S, _ = Q.shape

        Q = Q.reshape(B, h, S, self.d_head // 2, 2).unsqueeze(-2)
        Q = Q @ self.PE[:S]
        Q = Q.reshape(B, h, S, self.d_head)

        K = K.reshape(B, h, S, self.d_head // 2, 2).unsqueeze(-2)
        K = K @ self.PE[:S]
        K = K.reshape(B, h, S, self.d_head)

        return Q, K

In [ ]:
# Rotary Positional Encoding with Position Interpolation


class RotaryPositionalEncoding(nn.Module):
    def __init__(self, seq_len, d_head, scaling_factor):
        super().__init__()
        scaling_factor = max(1, scaling_factor)
        pos = torch.arange(seq_len).unsqueeze(1).float() / self.scaling_factor
        inv_freq = 1.0 / (10000 ** (torch.arange(0, d_head, 2).float() / d_head))
        cos = torch.cos(pos * inv_freq).repeat_interleave(2, dim=-1)
        sin = torch.sin(pos * inv_freq).repeat_interleave(2, dim=-1)
        self.register_buffer("cos", cos)
        self.register_buffer("sin", sin)

    def rotate_half(self, x):
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2 :]
        return torch.cat((-x2, x1), dim=-1)

    def applyRoPE(self, Q, K):
        B, h, S, _ = Q.shape

        Q = Q * self.cos[:S] + self.rotate_half(Q) * self.sin[:S]

        K = K * self.cos[:S] + self.rotate_half(K) * self.sin[:S]

        return Q, K

In [ ]:
# Rotary Positional Encoding with YaRN


class RotaryPositionalEncodingYaRN(nn.Module):
    def __init__(self, seq_len, d_head, scaling_factor, alfa, beta, original_seq_len):
        super().__init__()
        self.scaling_factor = max(1.0, scaling_factor)
        self.attention_temp = math.sqrt(1 + 0.1 * math.log(self.scaling_factor))
        pos = torch.arange(seq_len).unsqueeze(1).float()
        inv_freq = 1.0 / (10000 ** (torch.arange(0, d_head, 2).float() / d_head))
        ratio = original_seq_len * inv_freq / (2 * math.pi)
        gamma = torch.clamp((ratio - alfa) / (beta - alfa + 1e-5), 0.0, 1.0)
        inv_freq = torch.lerp(inv_freq / self.scaling_factor, inv_freq, gamma)

        cos = torch.cos(pos * inv_freq).repeat_interleave(2, dim=-1)
        sin = torch.sin(pos * inv_freq).repeat_interleave(2, dim=-1)

        self.register_buffer("cos", cos)
        self.register_buffer("sin", sin)

    def rotate_half(self, x):
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2 :]
        return torch.cat((-x2, x1), dim=-1)

    def applyRoPE(self, Q, K):
        B, h, S, _ = Q.shape

        Q = Q * self.cos[:S] + self.rotate_half(Q) * self.sin[:S]
        Q = Q / self.attention_temp

        K = K * self.cos[:S] + self.rotate_half(K) * self.sin[:S]
        K = K / self.attention_temp

        return Q, K

In [ ]:
# Layer Norm

class LayerNorm(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.beta = nn.Parameter(torch.zeros(d_model))
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.eps = 1e-5

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        std = x.std(dim=-1, keepdim=True)
        return self.beta + self.gamma * (x - mean) / (std + self.eps)


In [ ]:
# RMS Norm


class RMSNorm(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.eps = 1e-5

    def forward(self, x):
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return self.gamma * x / rms

In [ ]:
# Feed Forward Network

class FeedForwardNetwork(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.Wg = nn.Linear(d_model, d_ff, bias=False)
        self.Wu = nn.Linear(d_model, d_ff, bias=False)
        self.Wd = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        return self.Wd(nn.functional.silu(self.Wg(x)) * self.Wu(x))

In [ ]:
# Mixture of Experts Network


class MixtureOfExpertsNetwork(nn.Module):
    def __init__(self, d_model, d_ff, num_experts, top_k):
        super().__init__()
        self.top_k = top_k
        self.W_router = nn.Linear(d_model, num_experts)
        self.experts = nn.ModuleList(
            [FeedForwardNetwork(d_model, d_ff) for _ in range(num_experts)]
        )

    def forward(self, x):
        B, S, d = x.shape
        x = x.reshape(B * S, d)
        router_logits = self.W_router(x)
        router_probs = nn.functional.softmax(router_logits, dim=-1)
        topk_w, topk_idx = torch.topk(router_probs, self.top_k, dim=-1)
        weights = torch.zeros_like(router_probs)
        weights.scatter_(1, topk_idx, topk_w)
        weights = weights / weights.sum(dim=-1, keepdim=True)
        expert_outs = torch.stack(
            [expert(x) for expert in self.experts], dim=1
        ) 
        out = (weights.unsqueeze(-1) * expert_outs).sum(dim=1)  # (T, d)
        return out.reshape(B, S, d)

In [ ]:
# Cross Entropy Loss

class CrossEntropyLoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, y, logits):
        log_probs = nn.functional.log_softmax(logits, dim=-1)
        log_p = log_probs.gather(-1, y.unsqueeze(-1)).squeeze(-1)
        return (-log_p).mean()

In [ ]:
# SGD

class SGD:
    def __init__(self, params, lr):
        self.params = list(params)
        self.lr = lr

    def step(self):
        with torch.no_grad():
            for p in self.params:
                if p.grad is not None:
                    p -= self.lr * p.grad

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()

In [ ]:
# SGD + Momentum
class SGDMomentum:
    def __init__(self, params, lr, momentum):
        self.params = list(params)
        self.lr = lr
        self.momentum = momentum
        self.m = {}

    def step(self):
        with torch.no_grad():
            for p in self.params:
                if p.grad is None:
                    continue
                if p not in self.m:
                    self.m[p] = torch.zeros_like(p)
                self.m[p] = self.m[p] * self.momentum + p.grad
                p -= self.lr * self.m[p]

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()

In [ ]:
# Adam

class Adam:
    def __init__(self, params, lr, beta1, beta2):
        self.params = list(params)
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.t = 0
        self.eps = 1e-5
        self.m = {}
        self.v = {}

    def step(self):
        self.t += 1
        with torch.no_grad():
            for p in self.params:
                if p.grad is None:
                    continue
                if p not in self.m:
                    self.m[p] = torch.zeros_like(p)
                if p not in self.v:
                    self.v[p] = torch.zeros_like(p)
                self.m[p] = self.beta1 * self.m[p] + (1 - self.beta1) * p.grad
                self.v[p] = self.beta2 * self.v[p] + (1 - self.beta2) * p.grad * p.grad
                p -= (
                    self.lr
                    * (self.m[p] / (1 - self.beta1**self.t))
                    / (torch.sqrt((self.v[p] / (1 - self.beta2**self.t))) + self.eps)
                )

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()

In [ ]:
# AdamW

class AdamW:
    def __init__(self, params, lr, beta1, beta2, gamma):
        self.params = list(params)
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.gamma = gamma
        self.t = 0
        self.eps = 1e-5
        self.m = {}
        self.v = {}

    def step(self):
        self.t += 1
        with torch.no_grad():
            for p in self.params:
                if p.grad is None:
                    continue
                if p not in self.m:
                    self.m[p] = torch.zeros_like(p)
                if p not in self.v:
                    self.v[p] = torch.zeros_like(p)
                self.m[p] = self.beta1 * self.m[p] + (1 - self.beta1) * p.grad
                self.v[p] = self.beta2 * self.v[p] + (1 - self.beta2) * p.grad * p.grad
                p -= (
                    self.lr
                    * (self.m[p] / (1 - self.beta1**self.t))
                    / (torch.sqrt((self.v[p] / (1 - self.beta2**self.t))) + self.eps)
                ) + self.lr * self.gamma * p

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()

In [ ]:
import torch
import torch.nn as nn

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, S, d = x.shape

        Q = self.W_q(x)
        Q = Q.reshape(B, S, self.n_heads, self.d_head).transpose(1, 2)

        K = self.W_k(x)
        K = K.reshape(B, S, self.n_heads, self.d_head).transpose(1, 2)

        V = self.W_v(x)
        V = V.reshape(B, S, self.n_heads, self.d_head).transpose(1, 2)

        attention_scores = Q @ K.transpose(-1, -2) / (self.d_head ** 0.5)
        causal_mask = torch.triu(torch.ones(S, S, dtype=bool, device=x.device), diagonal=1)
        attention_scores.masked_fill_(causal_mask, float("-inf"))
        attention_weights = nn.functional.softmax(attention_scores, dim=-1)

        A = attention_weights @ V
        A = A.transpose(1, 2).reshape(B, S, d)

        return self.W_o(A)



In [ ]:
def train_step(model, batch_iter, optimizer, scheduler, accum_steps, device):
    model.train()
    optimizer.zero_grad()

    for i in range(accum_steps):
        x, y = next(batch_iter)
        x, y = x.to(device), y.to(device)

        logits = model(x)                                  # (B, T, vocab)
        loss = loss_fn(logits.view(-1, logits.size(-1)),
                       y.view(-1)) / accum_steps
        loss.backward()

    optimizer.step()
    scheduler.step()
    optimizer.zero_grad()

    return loss.item()

In [ ]:
import torch.nn.functional as F

# Attention

class Attention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        self.W = nn.Linear(d_model, 3*d_model, bias = False)
        self.Wo = nn.Linear(d_model, d_model)

    def forward(self, x):
        _, S, d = x.shape
        Q, K, V = torch.chunk(self.W(x), 3, dim=-1)

        attention_scores = Q @ K.transpose(-1, -2) / self.d_model ** 0.5

        mask = torch.tril(torch.ones(S, S)).bool()
        masked_scores = attention_scores.masked_fill(~mask, float("-inf"))

        attention_weights = F.softmax(masked_scores, dim=-1)
        A = attention_weights @ V

        return self.Wo(A)

In [ ]:
# MHA

class MHA(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.W = nn.Linear(d_model, 3*d_model, bias=False)
        self.Wo = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B, S, _ = x.shape

        Q, K, V = torch.chunk(self.W(x), 3, dim=-1)

        Q = Q.view(B, S, self.n_heads, self.d_head).transpose(1, 2)
        K = K.view(B, S, self.n_heads, self.d_head).transpose(1, 2)
        V = V.view(B, S, self.n_heads, self.d_head).transpose(1, 2)

        attention_scores = Q @ K.transpose(-1, -2) / self.d_head ** 0.5
        mask = torch.tril(torch.ones(S, S)).bool()
        masked_attention_scores = attention_scores.masked_fill(~mask, float("-inf"))

        attention_weights = F.softmax(masked_attention_scores, dim=-1)
        A = attention_weights @ V

        A = A.transpose(1, 2).contiguous().view(B, S, self.d_model)

        return self.Wo(A)




In [ ]:
# GQA


class GQA(nn.Module):
    def __init__(self, d_model, num_q_heads, num_kv_heads):
        super().__init__()
        self.d_model = d_model
        self.d_head = d_model // num_q_heads
        self.num_q_heads = num_q_heads
        self.num_kv_heads = num_kv_heads
        self.W = nn.Linear(
            d_model, (2 * num_kv_heads + num_q_heads) * self.d_head, bias=False
        )
        self.Wo = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B, S, _ = x.shape
        g = self.num_q_heads // self.num_kv_heads

        Q, K, V = torch.split(
            self.W(x),
            [
                self.num_q_heads * self.d_head,
                self.num_kv_heads * self.d_head,
                self.num_kv_heads * self.d_head,
            ],
            dim=-1,
        )

        Q = (
            Q.view(B, S, self.num_q_heads, self.d_head)
            .view(B, S, self.num_kv_heads, g, self.d_head)
            .permute(0, 2, 3, 1, 4)
        )

        K = (
            K.view(B, S, self.num_kv_heads, self.d_head)
            .view(B, S, self.num_kv_heads, 1, self.d_head)
            .permute(0, 2, 3, 1, 4)
        )

        V = (
            V.view(B, S, self.num_kv_heads, self.d_head)
            .view(B, S, self.num_kv_heads, 1, self.d_head)
            .permute(0, 2, 3, 1, 4)
        )

        attention_scores = Q @ K.transpose(-1, -2) / self.d_head**0.5
        mask = torch.tril(torch.ones(S, S)).bool()
        masked_attention_scores = attention_scores.masked_fill(~mask, float("-inf"))

        attention_weights = F.softmax(masked_attention_scores, dim=-1)

        A = attention_weights @ V
        A = (
            A.permute(0, 3, 1, 2, 4)
            .contiguous()
            .view(B, S, self.num_q_heads, self.d_head)
            .view(B, S, self.d_model)
        )

        return self.Wo(A)

In [ ]:
class AttentionWithCache(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        self.W = nn.Linear(d_model, 3 * d_model, bias=False)
        self.Wo = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, kv_cache=None):
        B, S, _ = x.shape

        Q, K, V = torch.chunk(self.W(x), 3, dim=-1)

        if kv_cache is not None:
            K_prev, V_prev = kv_cache
            K = torch.cat([K_prev, K], dim=1)
            V = torch.cat([V_prev, V], dim=1)

        new_cache = (K, V)

        attention_scores = Q @ K.transpose(-1, -2) / self.d_model**0.5
        masked_attention_scores = attention_scores
        if S > 1:
            mask = torch.tril(torch.ones(S, S)).bool()
            masked_attention_scores = attention_scores.masked_fill(~mask, float("-inf"))
        attention_weights = F.softmax(masked_attention_scores, dim=-1)
        A = attention_weights @ V

        return self.Wo(A), new_cache

In [ ]:
class GQAWithCache(nn.Module):
    def __init__(self, d_model, num_q_heads, num_kv_heads):
        super().__init__()
        self.d_model = d_model
        self.d_head = d_model // num_q_heads
        self.num_q_heads = num_q_heads
        self.num_kv_heads = num_kv_heads
        self.W = nn.Linear(
            d_model, (num_q_heads + 2 * num_kv_heads) * self.d_head, bias=False
        )
        self.Wo = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, kv_cache=None):
        B, S, _ = x.shape
        g = self.num_q_heads // self.num_kv_heads

        Q, K, V = torch.split(
            self.W(x),
            [
                self.num_q_heads * self.d_head,
                self.num_kv_heads * self.d_head,
                self.num_kv_heads * self.d_head,
            ],
            dim=-1,
        )

        if kv_cache is not None:
            K_prev, V_prev = kv_cache
            K = torch.cat([K_prev, K], dim=1)
            V = torch.cat([V_prev, V], dim=1)

        new_kv_cache = (K, V)

        Q = (
            Q.view(B, S, self.num_q_heads, self.d_head)
            .view(B, S, self.num_kv_heads, g, self.d_head)
            .permute(0, 2, 3, 1, 4)
        )

        K = (
            K.view(B, S, self.num_kv_heads, self.d_head)
            .view(B, S, self.num_kv_heads, 1, self.d_head)
            .permute(0, 2, 3, 1, 4)
        )

        V = (
            V.view(B, S, self.num_kv_heads, self.d_head)
            .view(B, S, self.num_kv_heads, 1, self.d_head)
            .permute(0, 2, 3, 1, 4)
        )

        attention_scores = Q @ K.transpose(-1, -2) / self.d_head**0.5

        if kv_cache is None:
            mask = torch.tril(torch.ones(S, S)).bool()
            attention_scores = attention_scores.masked_fill(~mask, float("-inf"))

        attention_weights = F.softmax(attention_scores, dim=-1)

        A = attention_weights @ V

        A = (
            A.permute(0, 3, 1, 2, 4)
            .contiguous()
            .view(B, S, self.num_q_heads, self.d_head)
            .view(B, S, self.d_model)
        )

        return self.Wo(A), new_kv_cache

In [ ]:
class MLA(nn.Module):
    def __init__(self, d_model, d_latent_q, d_latent_kv, d_rope, num_heads):
        super().__init__()
        self.d_model = d_model
        self.d_head = d_model // num_heads
        self.num_heads = num_heads
        self.d_latent_q = d_latent_q
        self.d_latent_kv = d_latent_kv
        self.d_rope = d_rope
        self.Wdq = nn.Linear(d_model, d_latent_q, bias=False)
        self.Wdkv = nn.Linear(d_model, d_latent_kv, bias=False)
        self.Wuq = nn.Parameter(torch.randn(d_latent_q, d_model))
        self.Wuk = nn.Parameter(torch.randn(d_latent_kv, d_model))
        self.Wuv = nn.Parameter(torch.randn(d_latent_kv, d_model))
        self.Wo = nn.Parameter(torch.randn(d_model, d_model))
        self.Wrq = nn.Linear(d_model, num_heads * d_rope, bias=False)
        self.Wrk = nn.Linear(d_model, d_rope, bias=False)

        self.register_buffer("W_abs", None)
        self.register_buffer("Wo_abs", None)

    @torch.no_grad()
    def _absorb(self):
        Wuq_h = self.Wuq.view(
            self.d_latent_q,
            self.num_heads,
            self.d_head,
        ).transpose(0, 1)

        Wuk_h = self.Wuk.T.contiguous().view(
            self.num_heads,
            self.d_head,
            self.d_latent_kv,
        )
        self.W_abs = Wuq_h @ Wuk_h

        Wuv_h = self.Wuv.view(self.d_latent_kv, self.num_heads, self.d_head).transpose(
            0, 1
        )
        Wo_h = self.Wo.view(self.num_heads, self.d_head, self.d_model)
        self.Wo_abs = (Wuv_h @ Wo_h).view(
            self.num_heads * self.d_latent_kv, self.d_model
        )

    def forward(self, x, kv_cache=None):
        B, S_q, _ = x.shape
        c_q = self.Wdq(x)
        c_kv = (
            self.Wdkv(x)
            if kv_cache is None
            else torch.cat([kv_cache[0], self.Wdkv(x)], dim=1)
        )
        S_kv = c_kv.shape[1]

        if self.W_abs is None or self.training:
            self._absorb()

        scores_content = (
            c_q.view(B, 1, S_q, self.d_latent_q)
            @ self.W_abs
            @ c_kv.transpose(-1, -2).contiguous().view(B, 1, self.d_latent_kv, S_kv)
        ) / self.d_head**0.5

        r_q = self.Wrq(x)
        r_q = r_q.view(B, S_q, self.num_heads, self.d_rope).transpose(1, 2)
        r_q = apply_rope(r_q)
        r_k = self.Wrk(x) if kv_cache is None else torch.cat([kv_cache[1], self.Wrk(x)], dim=1)
        r_k = r_k.view(B, 1, S_kv, self.d_rope)
        r_k = apply_rope(r_k)
        
        scores_rope = r_q @ r_k.transpose(-1, -2) / self.d_rope ** 0.5

        scores = scores_content + scores_rope

        if kv_cache is None:
            mask = torch.tril(torch.ones(S_q, S_q)).bool()
            scores = scores.masked_fill(~mask, float("-inf"))

        weights = F.softmax(scores, dim=-1) @ c_kv.view(B, 1, S_kv, self.d_latent_kv)
        weights = (
            weights.transpose(1, 2)
            .contiguous()
            .view(B, S_q, self.num_heads * self.d_latent_kv)
        )

        A = weights @ self.Wo_abs

        return A, (c_kv, r_k)

In [ ]:
# RoPE


class RotaryEmbedding(nn.Module):
    def __init__(self, d, base, max_seq_len):
        super().__init__()
        self.d = d
        self.base = base
        self.max_seq_len = max_seq_len

        inv_freq = 1 / base ** (torch.arange(0, d, 2) / d).float()
        pos = torch.arange(0, max_seq_len).float().unsqueeze(-1)
        PE = inv_freq * pos

        cos = torch.cos(PE)
        cos = torch.cat([cos, cos], dim=-1)
        sin = torch.sin(PE)
        sin = torch.cat([sin, sin], dim=-1)
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)

    def rotate_half(self, x):
        x1, x2 = x[..., : x.shape[-1] // 2], x[..., x.shape[-1] // 2 :]
        return torch.cat([-x2, x1], dim=-1)

    def apply_rope(self, x, position_ids, unsqueeze_dim=1):
        cos = self.cos[position_ids].unsqueeze(unsqueeze_dim)
        sin = self.sin[position_ids].unsqueeze(unsqueeze_dim)
        return cos * x + sin * self.rotate_half(x)

In [ ]:
# Mixture of Experts

class MoE(nn.Module):
    def __init__(self, num_experts, num_shared_experts, d_model):
        super().__init__()
        self.router = nn.Linear(d_model, num_experts)
        self.shared_experts = [nn.Linear(d_model, d_model) for _ in range(num_shared_experts)]
        self.experts = [nn.Linear(d_model, d_model) for _ in range(num_experts)]


    def forward(self, x):
        B, S, d_model = x.shape
        scores = torch.sigmoid(self.router(x))